In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ==============================================================================
# Step 1: Environment Setup & Dataset Initialization
# ==============================================================================

# Install the necessary libraries: Ultralytics (for YOLOv8) and Roboflow (for data)
!pip install ultralytics roboflow

# Clear the output to keep the notebook clean
from IPython.display import clear_output
clear_output()

print("Libraries installed successfully! ✅")

# Import the YOLO architecture
from ultralytics import YOLO

# Initialize Roboflow and download the dataset directly into the Colab environment
from roboflow import Roboflow

# Initialize with API Key (Keep this secret in production)
rf = Roboflow(api_key="RQMPMcpDeL2nFd04kTvI")

# Fetch the specific workspace, project, and version
project = rf.workspace("roadeye-workspace").project("roadeye-josak")
version = project.version(1)

# Download the dataset in YOLOv8 format
dataset = version.download("yolov8")

print("\nDataset downloaded successfully and ready for training! 🚀")
print(f"Dataset location: {dataset.location}")

Libraries installed successfully! ✅
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to RoadEye-1 in yolov8:: 100%|██████████| 16771/16771 [00:02<00:00, 7714.22it/s] 


Dataset downloaded successfully and ready for training! 🚀
Dataset location: /content/RoadEye-1


In [ ]:
# ==============================================================================
# Step 2: YOLOv8 Model Training Configuration (Secure Drive Version)
# ==============================================================================

import os
from ultralytics import YOLO

# Define the path to the dataset configuration file (data.yaml)
# Ensure this matches the folder name downloaded from Roboflow
dataset_yaml_path = '/content/RoadEye-1/data.yaml'

# Ensure the file exists before starting training to avoid crashes
if not os.path.exists(dataset_yaml_path):
    print(f"❌ Error: data.yaml not found at {dataset_yaml_path}")
else:
    print(f"✅ data.yaml found! Initializing secure model training...")

    # Load the YOLOv8 Nano model (Pre-trained on COCO dataset for faster convergence)
    model = YOLO('yolov8n.pt')

    # Define the SECURE path on your mounted Google Drive
    # This guarantees that weights and logs survive any Colab disconnections
    drive_project_path = '/content/drive/MyDrive/RoadEye_Model'

    # Start the training process
    results = model.train(
        data=dataset_yaml_path,     # Path to the dataset config
        epochs=50,                  # Maximum number of training iterations
        imgsz=640,                  # Image size (Must match Roboflow export size)
        batch=16,                   # Number of images per batch (Safe for Colab T4)
        patience=10,                # Early stopping criterion
        project=drive_project_path, # 🛡️ SAVING DIRECTLY TO GOOGLE DRIVE 🛡️
        name='v1_nano_balanced',    # Name of this specific training run
        device=0                    # 0 forces the use of the primary GPU
    )

    print(f"\n🎉 Training completed! Check your Google Drive at '{drive_project_path}/v1_nano_balanced' for weights and graphs.")

✅ data.yaml found! Initializing secure model training...
Ultralytics 8.4.33 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/RoadEye-1/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=v1_nano_balanced, nbs=64, nms=False, opset=None, optimize=Fal

In [ ]:
# ==============================================================================
# Step 3: YOLOv8 Medium Training (Targeting 70%+ mAP)
# ==============================================================================

print("🚀 Initializing Phase 2: High-Accuracy Model Training (YOLOv8 Medium)...")

# Load the Medium model for much higher accuracy
model_m = YOLO('yolov8m.pt')

# Start the training process
results_m = model_m.train(
    data='/content/RoadEye-1/data.yaml',
    epochs=100,                  # 🔥 Increased from 50 to 100
    imgsz=640,
    batch=16,                    # T4 GPU can handle batch 16 for Medium easily
    patience=20,                 # 🔥 Increased patience to allow more learning time
    project='/content/drive/MyDrive/RoadEye_Model',
    name='v2_medium_balanced',   # New folder name to avoid overwriting V1
    device=0
)

print("\n🎉 Medium Model Training completed!")

🚀 Initializing Phase 2: High-Accuracy Model Training (YOLOv8 Medium)...
Ultralytics 8.4.33 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/RoadEye-1/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=v2_medium_balanced, nbs=64, nms=False, opset=

In [ ]:
# ==============================================================================
# Phase 3.2: The Ultimate V3 Training Loop (Stable Batch Version)
# ==============================================================================

from ultralytics import YOLO

# Clean start to wipe out previous memorization and learn properly from scratch
model = YOLO('yolov8m.pt')

# Paths for dataset and Google Drive saving
dataset_path = '/content/RoadEye-1/data.yaml'
drive_path = '/content/drive/MyDrive/RoadEye_Model'

# Start the highly optimized training process
results = model.train(
    data=dataset_path,
    epochs=100,
    imgsz=640,
    batch=16,            # Kept at 16 for guaranteed T4 GPU stability (Zero OOM risk)
    patience=20,
    project=drive_path,
    name='v3_master_medium_stable',

    # 🛡️ Explicit Custom Augmentations (Tailored for road anomalies)
    mosaic=1.0,          # Mix 4 images together (crucial for detecting small potholes)
    degrees=10.0,        # Simulate camera tilt/bumps
    blur_p=0.1,          # Simulate motion blur from speeding cars
    copy_paste=0.1,      # Copy potholes and paste them elsewhere to increase frequency

    # ⚙️ Hyperparameter Tuning for Stability & Generalization
    cos_lr=True,         # Enable Cosine Annealing for a smooth accuracy landing
    lr0=0.005,           # Conservative starting learning rate to handle heavy augmentation
    lrf=0.01,            # Final LR fraction
    weight_decay=0.0005, # Prevent overfitting

    device=0             # Force GPU usage
)

print("\n🚀 V3 Master Training Started Safely!")

In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="RQMPMcpDeL2nFd04kTvI")
project = rf.workspace("roadeye-workspace").project("roadeye-josak")
version = project.version(2)
dataset = version.download("yolov8")


In [ ]:
# الكود ده هيعرضلك كل الملفات اللي كولاب شايفها جوه الفولدر ده
!ls -la /content/drive/MyDrive/RoadEye_Model/

ls: cannot access '/content/drive/MyDrive/RoadEye_Model/': No such file or directory
